# Bird Detection - Inference Demo
## How to Use This Notebook

This notebook is for running predictions AFTER you have trained the model.

### Steps:
1. Train the model first using `train.py`
2. Make sure `checkpoints/best_model.pth` exists
3. Run all cells in this notebook
4. Update the image path in the last cell to test your own bird images

### What This Notebook Does:
- Loads your trained model from disk
- Takes a bird image as input
- Outputs the top 5 predicted species with confidence scores

## Cell 1: Import Libraries
These are the Python packages we need:
- `torch`: PyTorch for running the neural network
- `PIL.Image`: For loading images from disk
- `torchvision.transforms`: For preprocessing images
- `models.model`: Our custom BirdCNN architecture

In [ ]:
# Import PyTorch for neural network operations
import torch

# Import PIL (Python Imaging Library) for loading images
from PIL import Image

# Import transforms for preprocessing images
from torchvision import transforms

# Import our custom model architecture
from models.model import get_model

print("All libraries imported successfully!")

## Cell 2: Load Trained Model

### What Happens Here:
1. Detect if GPU is available (faster inference)
2. Load the saved model weights from `checkpoints/best_model.pth`
3. Create a fresh model architecture
4. Load the trained weights into the architecture
5. Set model to evaluation mode (disables dropout)

### Important Terms:
- **State Dict**: A dictionary containing all model weights/parameters
- **Checkpoint**: A saved snapshot of the model during training
- **Eval Mode**: Disables regularization (dropout) for consistent predictions

In [ ]:
# ============================================================
# STEP 1: Detect the best device (GPU or CPU)
# ============================================================
# CUDA = NVIDIA GPU acceleration (much faster than CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# STEP 2: Load the saved checkpoint
# ============================================================
# A checkpoint contains:
# - model_state_dict: All the learned weights
# - best_acc: The validation accuracy when this model was saved
# - epoch: Which training epoch this was saved at
checkpoint = torch.load(
    "checkpoints/best_model.pth",  # Path to saved model
    map_location=device              # Load to same device it was saved from
)
print(f"Checkpoint loaded from epoch {checkpoint['epoch']}")
print(f"Best validation accuracy: {checkpoint['best_acc']:.4f}")

# ============================================================
# STEP 3: Create model and load weights
# ============================================================
# IMPORTANT: You must create the model with the SAME number of
# classes as during training!
model = get_model(num_classes=200)  # Change 200 to your number of species

# Load the trained weights into the model
# .load_state_dict() copies weights from checkpoint to model
model.load_state_dict(checkpoint["model_state_dict"])

# Move model to GPU if available
model.to(device)

# ============================================================
# STEP 4: Set model to evaluation mode
# ============================================================
# .eval() does two important things:
# 1. Disables dropout (uses all neurons instead of random subset)
# 2. Uses running statistics for batch normalization
# ALWAYS call .eval() before inference!
model.eval()

print("\nModel loaded and ready for inference!")

## Cell 3: Define Prediction Function

### How Prediction Works:
1. Load the image from disk
2. Resize to 224x224 pixels
3. Convert to tensor and normalize
4. Add batch dimension (model expects batches)
5. Run through model (forward pass)
6. Convert logits to probabilities using softmax
7. Get top 5 predictions

### What are Logits vs Probabilities?
- **Logits**: Raw scores from the model (e.g., [2.1, 0.5, -1.3, ...])
- **Probabilities**: After softmax (e.g., [0.75, 0.15, 0.02, ...])
- Softmax ensures all values sum to 1.0 (like percentages)

In [ ]:
def predict(image_path, model, transform, classes):
    """
    Predict the bird species in an image.
    
    Args:
        image_path (str): Path to the bird image file
        model: Trained BirdCNN model
        transform: Image preprocessing pipeline
        classes (list): List of species names (e.g., ['cardinal', 'robin', ...])
    
    Returns:
        str: The name of the predicted bird species
    """
    
    # --------------------------------------------------------
    # STEP 1: Load and preprocess the image
    # --------------------------------------------------------
    # Open image and convert to RGB (3 channels)
    # .convert('RGB') handles RGBA, grayscale, etc.
    image = Image.open(image_path).convert("RGB")
    
    # Apply the same transforms used during training
    # This converts PIL Image -> PyTorch Tensor and normalizes
    input_tensor = transform(image)
    
    # Add batch dimension: (3, 224, 224) -> (1, 3, 224, 224)
    # Model expects batches, even if we have just 1 image
    # unsqueeze(0) adds a dimension at position 0
    input_tensor = input_tensor.unsqueeze(0).to(device)
    
    # --------------------------------------------------------
    # STEP 2: Run inference (forward pass)
    # --------------------------------------------------------
    # torch.no_grad() disables gradient computation
    # - Saves memory during inference
    # - Speeds up computation
    with torch.no_grad():
        # Forward pass: image -> model -> logits
        output = model(input_tensor)
        
        # Convert logits to probabilities using softmax
        # Softmax formula: p(i) = exp(z_i) / sum(exp(z_j))
        # This ensures all probabilities sum to 1.0
        probabilities = torch.nn.functional.softmax(output, dim=1)
        
        # Get top 5 predictions
        # torch.topk returns the k largest values and their indices
        top5_prob, top5_idx = torch.topk(probabilities, 5)
    
    # --------------------------------------------------------
    # STEP 3: Display results
    # --------------------------------------------------------
    print(f"\nPredictions for: {image_path}")
    print("-" * 40)
    
    # Loop through top 5 predictions
    for i in range(5):
        # Get species name and probability
        species_name = classes[top5_idx[0][i]]
        confidence = top5_prob[0][i]
        # Print as percentage with 2 decimal places
        print(f"  {i+1}. {species_name}: {confidence:.2%}")
    
    # Return the top prediction (most confident)
    return classes[top5_idx[0][0]]

## Cell 4: Define Transform and Test

### Transform Pipeline:
The image must be preprocessed the SAME WAY as during training:
1. **Resize**: Scale to 224x224 pixels
2. **ToTensor**: Convert to PyTorch tensor (0-1 float range)
3. **Normalize**: Standardize using ImageNet statistics

### Note:
- We do NOT apply random augmentations during inference
- We want consistent, reproducible predictions

In [ ]:
# ============================================================
# Define the inference transform (NO augmentation)
# ============================================================
# This must match the validation/test transform from training!
transform = transforms.Compose([
    # Resize image to 224x224 pixels
    transforms.Resize((224, 224)),
    
    # Convert PIL Image to PyTorch Tensor
    # - Changes format from (H, W, C) to (C, H, W)
    # - Scales pixel values from [0, 255] to [0.0, 1.0]
    transforms.ToTensor(),
    
    # Normalize using ImageNet statistics
    # - These are the same values used during training
    # - mean and std for each RGB channel
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # RGB channel means
        std=[0.229, 0.224, 0.225]    # RGB channel standard deviations
    )
])

print("Transform pipeline ready!")

# ============================================================
# TEST: Uncomment the line below and update the image path
# ============================================================
# Replace 'path/to/your/bird_image.jpg' with your actual image path
# Example: predicted_species = predict("test_images/cardinal_001.jpg", model, transform, classes)
#
# You also need to load the class names:
# classes = checkpoint.get('classes', [f'class_{i}' for i in range(200)])
# predicted_species = predict("your_image.jpg", model, transform, classes)
# print(f"\nPredicted species: {predicted_species}")

## Quick Reference

### Common Issues:
1. **FileNotFoundError**: Make sure `checkpoints/best_model.pth` exists (run `train.py` first)
2. **Wrong number of classes**: Ensure `num_classes=200` matches your training config
3. **Different transform**: The transform must match what was used during training

### File Locations:
- Trained model: `checkpoints/best_model.pth`
- Model architecture: `models/model.py`
- Training script: `train.py`